# Generative AI: RAG & AI Agent

# Inferencig LLMs with LangChain

In [ ]:
!pip install -q langchain langchain_aws 

In [ ]:
from langchain_aws import ChatBedrock
llm = ChatBedrock(
    model_id="us.meta.llama3-3-70b-instruct-v1:0",
    model_kwargs=dict(temperature=0),
    region="us-east-1"
)

In [ ]:
response = llm.invoke("Who are you?")
print(response.content)

## Stateless

In [ ]:
response = llm.invoke("Hi! My name is Cisco")
print(response.content)

In [ ]:

response = llm.invoke("Hi! Do you remember my name")
print(response.content)

## Role - System Prompt

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a junior network engineer explaining concepts to a colleague who just started in networking. Avoid heavy technical jargon and keep answers short."),
    HumanMessage(content="What is a Router?")
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
messages = [
    SystemMessage(content="""You are a PhD in computer networking explaining concepts in a technically precise and academically rigorous manner. 
                             Include relevant protocols, layers, and theoretical context."""),
    HumanMessage(content="What is a Router?")
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
response = llm.invoke("What is the latest version of the Cisco ACI")
print(response.content)

# LangChain Chains

## Simple Chain

In [ ]:
!pip install -q langchain langchain-core

In [ ]:
from langchain_aws import ChatBedrock
llm = ChatBedrock(
    model_id="us.meta.llama3-3-70b-instruct-v1:0",
    model_kwargs=dict(temperature=0),
    region="us-east-1"
)

In [ ]:
from langchain_core.prompts.prompt import PromptTemplate

template = """
From the following Syslog message extract and explain:
1. Event type
2. Affected BGP peer
3. Probable cause
4. Recommended troubleshooting steps

Syslog message:
{message}
"""

prompt = PromptTemplate.from_template(template)


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

chain = prompt | llm

syslog_bgp = "*Aug  8 17:42:13.123: %BGP-5-ADJCHANGE: neighbor 192.0.2.1 Down BGP Notification sent"

messages = [
    SystemMessage(content="You are a senior network engineer analyzing Cisco IOS-XE syslog messages."),
    HumanMessage(content=prompt.format(message=syslog_bgp))
]

result = chain.invoke(messages)
print(result.content)

## Chain with JSON Parser

In [ ]:
from langchain_core.output_parsers.json import SimpleJsonOutputParser
import json

parser = SimpleJsonOutputParser()

template = """
You are a senior network engineer.

Extract the following fields from the syslog message and return ONLY valid JSON:
- timestamp
- event_type
- affected_ip
- reason

Syslog message:
{message}

{format_instructions}
"""
prompt = PromptTemplate.from_template(
    template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = llm | parser

messages = [
    SystemMessage(content="You are a senior network engineer analyzing Cisco IOS-XE syslog messages."),
    HumanMessage(content=prompt.format(message=syslog_bgp))
]

result = chain.invoke(messages)

print(json.dumps(result, indent=2))

## Multiple Chains

In [ ]:
parser_syslog = SimpleJsonOutputParser()

syslog_prompt_template = """
You are a senior network engineer.

Extract the following fields from the syslog message and return ONLY valid JSON:
- timestamp
- event_type
- affected_ips (List of devices/IPs/node names)
- reason

Syslog message:
{message}
"""
syslog_prompt = PromptTemplate.from_template(
    syslog_prompt_template,
)

syslog_chain = syslog_prompt | llm | parser_syslog

itsm_prompt_template = """
You are an IT support engineer.

Create an ITSM incident ticket in JSON format with the following fields:
- title (short summary of the event)
- date: (Time the even took place)
- affected devices: (List of affected devices, or IPs)
- description (detailed context)
- troubleshooting_steps (array of step-by-step actions)
- severity
- assigned team
- status

Here is the parsed syslog data:
{syslog_json}
"""
parser_itsm = SimpleJsonOutputParser()

itsm_prompt = PromptTemplate.from_template(
    itsm_prompt_template,
)

itsm_chain = itsm_prompt | llm | parser_itsm

# Combined chain: run syslog_chain → feed result into itsm_chain
combined_chain = {
    "syslog_json": syslog_chain  # this runs first
} | itsm_chain                   # output goes here

# Example syslog
syslog_bgp = "*Aug  8 17:13:02.456: %CRYPTO-4-IKMP_BAD_AUTH: IKE authentication between local address 198.51.100.10 and peer 203.0.113.5 failed. Possible pre-shared key"

ticket = combined_chain.invoke({"message": syslog_bgp})

print(json.dumps(ticket, indent=2))